# Metrics stage tour
A hands-on tour of the Metrics stage (`galactic_tidal_constraints.simr.metrics`) and of what feeds it:
externals (arrays from outside the run, such as a target) and the Derived stage
(`galactic_tidal_constraints.simr.derived`: arrays made from other arrays, such as weights). Run the
cells in order.

- **A metric** reduces its inputs to something small: a scalar method returns one float, a descriptive
  method a dict of declared keys.
- **A derived method** returns an array about the size of its input: a window of weights, a weight
  image, an edge map. A metric never takes another metric.

The first half calls each method directly on real arrays; "Metrics through `simr`" and the later
sections run them from config entries.

In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import metrics
from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

# Model SDSS 587722984435351614 (as in the other tours), and a copy with a heavier secondary.
params = np.array(
    [-9.93853, -4.5805, 15.43348, -1.54025, -2.97899, 4.69234, 34.41671, 29.49607, 2.97913, 4.2501,
     301.0, 35.5, 310.80745, 321.98757]
)
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2

## The metrics table
Each method is registered with the `metric` decorator where it's defined, which checks the declaration
at import and records it in `metrics.METHODS` — one source of truth. `metrics.table()` is the same
information as plain data: kind, input kinds, settings and defaults, direction (`best`) and `range` for
scalars, output keys and shapes for descriptors (`[]` is one float).

In [ ]:
print(json.dumps(metrics.table(), indent=2))

## `closest_approach`: a descriptive metric
`sim.search_orbit` is the secondary's center from now back to t = −30 (the path SPAM integrates backward
to place a run's start; a `simr` item since step 1). `closest_approach` finds the point along whatever
orbit it's given where the two centers are closest, and returns everything about it in one dict.

In [ ]:
path = sim.search_orbit(params).astype(np.float32)  # as the simr item hands it out
approach = metrics.closest_approach(path)
approach

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(path[:, 0], path[:, 1], lw=0.8, label="search orbit (t = 0 back to \u221230)")
ax.plot(0, 0, "k+", ms=12, label="primary (origin)")
ax.plot(path[0, 0], path[0, 1], "s", color="tab:green", label="now (t = 0)")
ax.plot(*approach["position"][:2], "o", color="tab:red",
        label=f"closest: d = {approach['distance']:.2f} at t = {approach['t']:.2f}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
ax.legend()
ax.set_title("closest_approach on sim.search_orbit")
plt.show()

## The entry's input decides the window
The method itself has no window: it reports the smallest distance in whatever orbit it's given. On
`sim.search_orbit` that's the whole window back to t = −30; on `sim.orbit` it's the run itself, so the
simulator config's `start` decides. Here the run starts before the most recent pass, so both find the
same pass; a run started later, or a window with an older pass, would differ.

In [ ]:
run_orbit = sim.simulate_orbit(params).astype(np.float32)
print("sim.search_orbit:", {k: approach[k] for k in ("t", "distance")})
print("sim.orbit:       ", {k: metrics.closest_approach(run_orbit)[k] for k in ("t", "distance")})

## `select`: lifting one float out of a descriptor
Composites (a later step) take scalars only, so a config entry can run a descriptive method and keep one
single-float key. Only keys with shape `()` qualify (`Metric.scalar_outputs`); naming an array-valued
key like `position` is a config error caught before anything runs. An entry looks like:
```json
"sim_metrics": [
  {"method": "closest_approach", "inputs": ["sim.search_orbit"]},
  {"method": "closest_approach", "inputs": ["sim.search_orbit"], "select": "distance"}
]
```

In [ ]:
print("scalar_outputs:", metrics.METHODS["closest_approach"].scalar_outputs)
print("select 'distance':", approach["distance"])
print("select 't':       ", approach["t"])

## `absolute_difference`: a scalar metric on two images
The mean |a − b| over the pixels of two greyscale images of the same shape: 0 is identical, and any two
matching images work, wherever they come from. Model vs unperturbed reads as how perturbed the model is;
against an external target (see Externals, below) it reads as how far from the target. Images already
flow through `simr`, so both come from one run (one default image entry; `data` is indexed
`[model, simulator, image]`).

In [ ]:
result = simr.run(np.stack([params, heavier]), {"run": {"return": ["img.model", "img.unperturbed"]}})
differences = [
    metrics.absolute_difference(result.data["img.model"][m, 0, 0], result.data["img.unperturbed"][m, 0, 0])
    for m in range(2)
]
differences

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for m, name in enumerate(["model", "heavier secondary"]):
    axes[m, 0].imshow(result.data["img.model"][m, 0, 0], cmap="gray")
    axes[m, 0].set_title(f"{name}: img.model")
    axes[m, 1].imshow(result.data["img.unperturbed"][m, 0, 0], cmap="gray")
    axes[m, 1].set_title(f"img.unperturbed \u2192 difference {differences[m]:.4f}")
for ax in axes.flat:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Mismatched shapes are an error
Matching image frames is the caller's job (no resize or shift inside the stage): two images of different
shapes are an error, not a silent crop. For an external image, `simr` catches this before anything runs
(see The checks, at the end).

In [ ]:
try:
    metrics.absolute_difference(np.zeros((128, 128), np.float32), np.zeros((64, 64), np.float32))
except ValueError as error:
    print("ValueError:", error)

## Metrics through `simr` (step 2b)
The `sim_metrics` and `img_metrics` config sections list entries (`method`, `inputs`, optional `select`
and `params`), checked against the metrics table before anything runs. Every `sim_metrics` entry runs on
every simulation and every `img_metrics` entry on every (simulation, image) pair, and the values always
come back: `result.sim_metrics[model, simulator, entry]` and `result.img_metrics[model, simulator, image,
entry]`, with per-cell status, time and error. `simr` computes what the entries need even when nothing
else asks for it — this request returns no data at all, only metrics.

In [ ]:
request = {
    "sim_metrics": [
        {"method": "closest_approach", "inputs": ["sim.search_orbit"]},
        {"method": "closest_approach", "inputs": ["sim.search_orbit"], "select": "distance"},
    ],
    "img_metrics": [{"method": "absolute_difference", "inputs": ["img.model", "img.unperturbed"]}],
}
metric_run = simr.run(np.stack([params, heavier]), request)
print("data returned:", metric_run.data)
print("sim_metrics", metric_run.sim_metrics.shape, "| img_metrics", metric_run.img_metrics.shape)
print("closest approach of model 0:", metric_run.sim_metrics[0, 0, 0])
print("select 'distance' per model:", metric_run.sim_metrics[:, 0, 1])
print("image difference per model: ", metric_run.img_metrics[:, 0, 0, 0])
print("filled-in entry 1:", metric_run.sim_metric_configs[1])
print("status:", metric_run.status["sim_metrics"].tolist())

## PCA: a metric for each particle input
`pca_shape` describes a particle cloud by its covariance ellipsoid (3D principal component analysis), on
the whole array as given: for `sim.particles` that's both galaxies together. `axes` are the semi-axes
(standard deviation along each principal axis), largest first; `vectors` the matching unit vectors as
rows, each with its largest component positive so the output is deterministic; `b_a` and `c_a` the axis
ratios (single floats, so `select` can pick them). Settings: `columns` (`"position"` or `"velocity"`),
`center` (`"mean"` or `"median"`) and `trim_quantile` (off by default: tidal tails are the outliers).

The plots show each cloud in x–y with its three axes at ±2σ, projected.

In [ ]:
# A different seed for each model, so their disks don't start from the same particle pattern: two
# simulator entries (seeds 1 and 2), taking model m from entry m. data is [model, simulator].
request = {"simulator": [{"seed": 1}, {"seed": 2}], "run": {"return": ["sim.particles", "sim.unperturbed"]}}
particle_run = simr.run(np.stack([params, heavier]), request)
clouds = [{item: particle_run.data[item][m, m] for item in ("sim.particles", "sim.unperturbed")} for m in range(2)]
shapes = [{item: metrics.pca_shape(cloud) for item, cloud in pair.items()} for pair in clouds]
print("shape:", clouds[0]["sim.particles"].shape)
shapes[0]["sim.particles"]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for m, model in enumerate(["model", "heavier secondary"]):
    for k, item in enumerate(["sim.particles", "sim.unperturbed"]):
        ax, cloud, shape = axes[m, k], clouds[m][item], shapes[m][item]
        ax.scatter(cloud[:, 0], cloud[:, 1], s=0.2, color="0.4")
        for axis, vector, color in zip(shape["axes"], shape["vectors"], ["tab:red", "tab:blue", "tab:green"]):
            ends = shape["center"][:2] + np.outer([-1, 1], 2 * axis * vector[:2])  # +-2 sigma
            ax.plot(ends[:, 0], ends[:, 1], color=color, lw=2)
        ax.set_title(f"{model}: {item}\nb/a {shape['b_a']:.2f}, c/a {shape['c_a']:.2f}")
        ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### `pca_difference`: a scalar metric on two particle clouds
How different two clouds' shapes are, from their ellipsoids: 0 is the same shape, and the clouds may
hold different numbers of particles. The default compares the axis ratios `[b/a, c/a]` with an L2
distance, blind to size, position and rotation (values in [0, √2]). `descriptor="axes"` lets size count,
`"geometric"` compares `[linearity, planarity, sphericity]`, and `orientation_weight` w adds
w·(1 − |v1_a·v1_b|) for how far apart the longest axes point (SPAM's frame is the sky's, so orientation
is real information). Model vs unperturbed reads as how much the encounter changed the system's shape.

In [ ]:
for name, settings in [
    ("ratios, l2 (default)", {}),
    ("ratios, l1", {"distance": "l1"}),
    ("axes (size counts)", {"descriptor": "axes"}),
    ("geometric", {"descriptor": "geometric"}),
    ("ratios + orientation (w = 1)", {"orientation_weight": 1.0}),
    ("velocities", {"columns": "velocity"}),
]:
    values = [metrics.pca_difference(clouds[m]["sim.particles"], clouds[m]["sim.unperturbed"], **settings)
              for m in range(2)]
    print(f"{name:30s} model {values[0]:.4f}   heavier {values[1]:.4f}")

### All four particle sets against each other
Every pair (rows vs columns), one table per descriptor: 0 on the diagonal, and the off-diagonal cells show
which sets have similar shapes.

In [ ]:
sets = {
    "model / perturbed": clouds[0]["sim.particles"],
    "model / unperturbed": clouds[0]["sim.unperturbed"],
    "heavier / perturbed": clouds[1]["sim.particles"],
    "heavier / unperturbed": clouds[1]["sim.unperturbed"],
}
names = list(sets)
for descriptor in metrics.PCA_DESCRIPTORS:
    grid = np.array([[metrics.pca_difference(sets[a], sets[b], descriptor=descriptor) for b in names] for a in names])
    print(f"descriptor={descriptor!r} (rows vs columns, 0 = same shape)")
    print(" " * 24 + "".join(f"{n:>24s}" for n in names))
    for name, row in zip(names, grid):
        print(f"{name:24s}" + "".join(f"{v:24.4f}" for v in row))
    print()

### PCA metrics through `simr`
Like any method, the PCA pair runs from config entries; settings go in `params`, and the filled-in entry
records every setting with its default.

In [ ]:
request = {
    "sim_metrics": [
        {"method": "pca_shape", "inputs": ["sim.particles"], "select": "c_a"},
        {"method": "pca_difference", "inputs": ["sim.particles", "sim.unperturbed"]},
        {"method": "pca_difference", "inputs": ["sim.particles", "sim.unperturbed"],
         "params": {"orientation_weight": 1.0}},
    ],
}
pca_run = simr.run(np.stack([params, heavier]), request)
for e, entry in enumerate(pca_run.sim_metric_configs):
    print(entry["method"], entry["params"], "->", pca_run.sim_metrics[:, 0, e].tolist())

## Externals: arrays from outside the run
A metric usually compares a model with a **target**. Targets aren't made by `simr`, so they come with
the run, like the models: `simr.run(models, config, externals={...})`. In the config an external is
named `ext.<key>`. Every input name has a prefix that says where it comes from:

| Prefix | Comes from |
|---|---|
| `sim.` | The simulator |
| `img.` | The Image Creator |
| `drv.` | The `derived` section (below) |
| `ext.` | Outside the run: the `externals` given with it |

There's no TNG data here, so the stand-in target is a SPAM run of one model (`mass2` × 1.5) with another
seed. Five models, differing only in the secondary's mass, are compared with it.

In [ ]:
import logging
import tempfile
from pathlib import Path

from galactic_tidal_constraints.simr import derived


def with_mass2(factor):
    model = params.copy()
    model[sim.PARAM_INDEX["mass2"]] *= factor
    return model


factors = [1.0, 1.25, 1.5, 1.75, 2.0]
models = np.stack([with_mass2(factor) for factor in factors])
simulator = {"n1": 2000, "n2": 1000}

made = simr.run(
    with_mass2(1.5),
    {"run": {"return": ["sim.search_orbit", "sim.particles", "img.model", "img.unperturbed"]},
     "simulator": {**simulator, "seed": 1}},
)
target_orbit = made.data["sim.search_orbit"][0, 0][::-1][::50].copy()  # increasing time, every 0.05
target_particles = made.data["sim.particles"][0, 0]
target_image = made.data["img.model"][0, 0, 0]
unperturbed_image = made.data["img.unperturbed"][0, 0, 0]
print("target orbit:    ", target_orbit.shape, " times", target_orbit[0, 6], "to", target_orbit[-1, 6])
print("target particles:", target_particles.shape)
print("target image:    ", target_image.shape)

In [ ]:
request = {
    "run": {"return": ["img.model"]},
    "simulator": simulator,
    "sim_metrics": [{"method": "pca_difference", "inputs": ["ext.target_particles", "sim.particles"]}],
    "img_metrics": [{"method": "absolute_difference", "inputs": ["ext.target_image", "img.model"]}],
}
externals = {"target_particles": target_particles, "target_image": target_image}
ext_run = simr.run(models, request, externals=externals)

for factor, pca, pixels in zip(factors, ext_run.sim_metrics[:, 0, 0], ext_run.img_metrics[:, 0, 0, 0]):
    print(f"mass2 x {factor:<4}   pca_difference {pca:.4f}   absolute_difference {pixels:.4f}")

fig, axes = plt.subplots(1, 6, figsize=(18, 3.4))
axes[0].imshow(target_image, cmap="gray", origin="lower")
axes[0].set_title("target (external)")
for ax, factor, image, pixels in zip(axes[1:], factors, ext_run.data["img.model"][:, 0, 0],
                                     ext_run.img_metrics[:, 0, 0, 0]):
    ax.imshow(image, cmap="gray", origin="lower")
    ax.set_title(f"mass2 x {factor}: {pixels:.4f}")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

### What gets recorded
Each external a metric uses is converted to `float32`, checked, and **hashed**
(`simr.array_hash`: SHA-256 of its shape and values), before anything is simulated. Its source, shape
and hash come back in `result.externals` and go in the run's record in `catalog.json`, so a run says
which data it used. The same data gives the same hash from memory or from a file.

A value can also be the absolute path of a `.npy` file, optionally with the hash its data must have.

In [ ]:
print(json.dumps(ext_run.externals, indent=2))

folder = Path(tempfile.mkdtemp())
np.save(folder / "target_image.npy", target_image)
pinned = {"target_particles": target_particles,
          "target_image": {"path": folder / "target_image.npy", "hash": ext_run.externals["target_image"]["hash"]}}
from_file = simr.run(models[:1], request, output=folder / "runs", externals=pinned)
print("from a file, same hash:", from_file.externals["target_image"]["hash"] == ext_run.externals["target_image"]["hash"])
print("in the catalog:", list(simr.load_catalog(folder / "runs")["runs"][0]["externals"]))

np.save(folder / "target_image.npy", target_image * 0.5)  # the file changes after its hash was taken
try:
    simr.run(models[:1], request, externals=pinned)
except ValueError as error:
    print("file changed -> ValueError:", str(error)[:110], "...")

With a pool (`map=pool.map`), the workers share **one** copy of each external through `/dev/shm`, a
folder in the node's memory, so nothing is copied per model. That makes a pool with in-memory externals
single-node; `float32` `.npy` files on a disk every node reads work across nodes. See Externals in
`docs/configuration.md`.

## The Derived stage: arrays made from other arrays
A derived method returns an array about the size of its input, for metrics to use as **data** or as
**weights**. Like the metrics, each is registered with its input kinds, the kind it returns and its
settings.

In [ ]:
print(json.dumps(derived.table(), indent=2))

### `recent_approach_cosine_window`: weights along an orbit
The top panel is the distance between the two galaxies along the target orbit. A **pass** is where it
stops falling and starts rising; the window's peak is the most recent one. The window is 1 there and
falls as a cosine to 0 at now (t = 0) and at `start = peak − k × (0 − peak)`.

- `k` stretches only the early side.
- `flat` keeps a fraction of each side at 1 before the cosine starts.
- `pass_number` picks an older pass, if the window still fits before the orbit's first time. A window
  that doesn't fit is an error, not a clipped window.

In [ ]:
t = target_orbit[:, 6]
distance = np.linalg.norm(target_orbit[:, :3], axis=1)
passes = np.flatnonzero((distance[1:-1] < distance[:-2]) & (distance[1:-1] <= distance[2:])) + 1
print("passes at t =", np.round(t[passes], 2), " (the most recent is the last)")

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(t, distance, color="k")
axes[0].plot(t[passes], distance[passes], "o", color="tab:red", label="passes")
axes[0].set_ylabel("separation")
axes[0].legend()
for k in (0.5, 1.0, 2.0):
    axes[1].plot(t, derived.recent_approach_cosine_window(target_orbit, k=k), label=f"k = {k}")
axes[1].set_ylabel("weight")
axes[1].set_title("k: how far back the window reaches (flat = 0)")
axes[1].legend()
for flat in (0.0, 0.5, 1.0):
    axes[2].plot(t, derived.recent_approach_cosine_window(target_orbit, k=1.0, flat=flat), label=f"flat = {flat}")
axes[2].set_ylabel("weight")
axes[2].set_title("flat: the fraction of each side held at 1 (k = 1)")
axes[2].set_xlabel("time")
axes[2].legend()
axes[2].set_xlim(-12, 0.5)
plt.tight_layout()
plt.show()

for settings in ({"pass_number": 2, "k": 0.05}, {"pass_number": 2, "k": 1.0}, {"k": 20.0}):
    try:
        weights = derived.recent_approach_cosine_window(target_orbit, **settings)
        print(settings, "-> peak at t =", round(float(t[np.argmax(weights)]), 2))
    except ValueError as error:
        print(settings, "-> ValueError:", error)

### `difference_weights` and `detect_edges`: image-shaped arrays
`difference_weights(a, b)` is `|a − b|`, scaled so its largest value is 1, with a small `floor` so
regions where the images agree still count a little. With the target and an **unperturbed** image, it's
large where the target departs from undisturbed disks; neither image is the model's, so a model can't
shrink its own weights.

`detect_edges(image)` is the Sobel edge strength, scaled so the strongest edge is 1.

Both take an optional `threshold` that turns the result into an on/off mask.

In [ ]:
panels = [
    ("target", target_image),
    ("unperturbed", unperturbed_image),
    ("difference_weights\n(floor 0.05)", derived.difference_weights(target_image, unperturbed_image)),
    ("difference_weights\n(threshold 0.3)", derived.difference_weights(target_image, unperturbed_image, threshold=0.3)),
    ("detect_edges(target)", derived.detect_edges(target_image)),
    ("detect_edges\n(threshold 0.3)", derived.detect_edges(target_image, threshold=0.3)),
]
fig, axes = plt.subplots(1, 6, figsize=(19, 3.6))
for ax, (title, image) in zip(axes, panels):
    ax.imshow(image, cmap="gray", origin="lower", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Weights, and the orbit metrics
A metric that can use weights (`weights: true` in the metrics table) takes them as one more input after
its own. Without them every row or pixel counts equally. Weights always match the method's **first**
input: one value per row of an orbit or of particles, or an image of the same shape.

The two orbit metrics compare a **target** orbit (first) with a **model's** orbit (second), in SPAM
length units squared, best low:

- `orbit_time_error`: how far apart the two are **at the same times**.
- `orbit_position_error`: how far the target's **path** is from the model's path, whatever the timing.

Model 2 (`mass2` × 1.5) is the target's own model, so it scores 0.

In [ ]:
paths = [sim.search_orbit(model, sim.SimulatorConfig(**simulator)).astype(np.float32) for model in models]
window = derived.recent_approach_cosine_window(target_orbit, k=1.0)

print(f"{'mass2 x':>8} {'time, even':>12} {'time, window':>13} {'position, even':>15} {'position, window':>17}")
for factor, model_path in zip(factors, paths):
    print(f"{factor:>8} {metrics.orbit_time_error(target_orbit, model_path):>12.4f} "
          f"{metrics.orbit_time_error(target_orbit, model_path, window):>13.4f} "
          f"{metrics.orbit_position_error(target_orbit, model_path):>15.4f} "
          f"{metrics.orbit_position_error(target_orbit, model_path, window):>17.4f}")

"Even" weights count the whole target orbit, back to t = −30, where the models have drifted far apart;
the window counts only the stretch around the most recent pass. The orbits in that stretch, looking
down z:

In [ ]:
inside = window > 0
fig, ax = plt.subplots(figsize=(6.5, 6.5))
for factor, model_path in zip(factors, paths):
    rows = (model_path[:, 6] >= t[inside][0]) & (model_path[:, 6] <= t[inside][-1])
    ax.plot(model_path[rows, 0], model_path[rows, 1], label=f"model, mass2 x {factor}", lw=1.2)
ax.scatter(target_orbit[inside, 0], target_orbit[inside, 1], c=window[inside], cmap="viridis", s=28,
           zorder=3, label="target rows (color = weight)")
ax.plot(0, 0, "k+", ms=12)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_title("Inside the window: target rows and each model's path")
plt.tight_layout()
plt.show()

**The target's spacing matters little.** Each row stands for the time around it (the trapezoid rule),
so the value is an average over time, not over rows. Below, the same target sampled three ways. The
values agree to a few percent, not exactly: the window's peak is taken at a target row, with no
interpolation, so a coarser target puts the peak (and the window's start) at a slightly different
time. The last column is what a plain average over rows would give: the same when the rows are evenly
spaced, and different when they bunch up, as in the third line.

In [ ]:
dense = made.data["sim.search_orbit"][0, 0][::-1]  # every 0.001
samplings = {
    "every 0.05 (as above)": dense[::50],
    "every 0.2": dense[::200],
    "uneven: every 0.01 from t = -3.5 to -2.2, every 0.25 elsewhere": np.concatenate(
        [dense[dense[:, 6] < -3.5][::250], dense[(dense[:, 6] >= -3.5) & (dense[:, 6] < -2.2)][::10],
         dense[dense[:, 6] >= -2.2][::250]]),
}
heavier_path = paths[4][::-1]  # the heaviest model, in increasing time
for label, sampled in samplings.items():
    sampled = np.ascontiguousarray(sampled)
    weights = derived.recent_approach_cosine_window(sampled, k=1.0)
    rows = weights > 0
    nearest = np.searchsorted(heavier_path[:, 6], sampled[rows, 6]).clip(0, len(heavier_path) - 1)
    by_row = float(np.average(((sampled[rows, :3] - heavier_path[nearest, :3]) ** 2).sum(axis=1), weights=weights[rows]))
    print(f"{len(sampled):>5} rows, {label:<62} peak at t = {sampled[np.argmax(weights), 6]:.2f}   "
          f"time {metrics.orbit_time_error(sampled, paths[4], weights):.4f}   "
          f"position {metrics.orbit_position_error(sampled, paths[4], weights):.4f}   "
          f"(plain average over rows: {by_row:.4f})")

## All of it from one config
The `derived` section lists named entries, in order; metrics use one as `drv.<name>`, as weights or as
data. An entry may use entries above it, so a loop is impossible.

In [ ]:
config = {
    "run": {"return": ["drv.target_win", "drv.tidal"]},
    "simulator": simulator,
    "derived": [
        {"name": "target_win",   "method": "recent_approach_cosine_window", "inputs": ["ext.target_orbit"],
         "params": {"k": 1.0}},
        {"name": "tidal",        "method": "difference_weights", "inputs": ["ext.target_image", "img.unperturbed"]},
        {"name": "target_edges", "method": "detect_edges", "inputs": ["ext.target_image"]},
        {"name": "model_edges",  "method": "detect_edges", "inputs": ["img.model"]},
    ],
    "sim_metrics": [
        {"method": "orbit_time_error",     "inputs": ["ext.target_orbit", "sim.search_orbit", "drv.target_win"]},
        {"method": "orbit_position_error", "inputs": ["ext.target_orbit", "sim.search_orbit", "drv.target_win"]},
    ],
    "img_metrics": [
        {"method": "absolute_difference", "inputs": ["ext.target_image", "img.model"]},
        {"method": "absolute_difference", "inputs": ["ext.target_image", "img.model", "drv.tidal"]},
        {"method": "absolute_difference", "inputs": ["drv.target_edges", "drv.model_edges"]},
    ],
}
targets = {"target_orbit": target_orbit, "target_image": target_image}
full_run = simr.run(models, config, externals=targets)

print(f"{'mass2 x':>8} {'orbit time':>11} {'orbit position':>15} | {'image':>8} {'image, tidal weights':>21} {'edges':>8}")
for m, factor in enumerate(factors):
    time_error, position_error = full_run.sim_metrics[m, 0]
    plain, weighted, edges = full_run.img_metrics[m, 0, 0]
    print(f"{factor:>8} {time_error:>11.4f} {position_error:>15.4f} | {plain:>8.4f} {weighted:>21.4f} {edges:>8.4f}")

**How often each entry is computed** follows from what it's made from (its `level`): once for the run
(externals only), once per simulation (any `sim.*` item) or once per image (any `img.*` item). One
computed once is handled like an external from then on, and its shape and hash are recorded. An entry
adds one array per run, simulation or image, and no new combinations. `run.return` hands them back
indexed like the data they were made from.

In [ ]:
for entry in full_run.derived_configs:
    extra = f"  shape {entry['shape']}, hash {entry['hash'][:12]}..." if "hash" in entry else ""
    print(f"drv.{entry['name']:<13} level {entry['level']!r:<6} {entry['method']}({', '.join(entry['inputs'])}){extra}")
print()
print("drv.target_win:", type(full_run.data["drv.target_win"]).__name__, full_run.data["drv.target_win"].shape,
      "- one array, the same for every model")
print("drv.tidal:     ", full_run.data["drv.tidal"].shape, "- [model, simulator, image], one weight image each")

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, factor, weights in zip(axes, factors, full_run.data["drv.tidal"][:, 0, 0]):
    ax.imshow(weights, cmap="gray", origin="lower", vmin=0, vmax=1)
    ax.set_title(f"drv.tidal, mass2 x {factor}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## The checks
Everything about the config and the externals is checked before anything is simulated, and each error
says where the problem is.

In [ ]:
def show(label, change, externals=targets):
    try:
        simr.run(models[:1], {**config, **change}, externals=externals)
        print(f"{label}:\n  ran\n")
    except ValueError as error:
        print(f"{label}:\n  ValueError: {error}\n")


orbit_metric = config["sim_metrics"][0]
show("no externals given", {}, externals=None)
show("an external named without its ext. prefix",
     {"sim_metrics": [{**orbit_metric, "inputs": ["target_orbit", "sim.search_orbit"]}]})
show("a typo in an item's name",
     {"sim_metrics": [{**orbit_metric, "inputs": ["ext.target_orbit", "sim_search_orbit"]}]})
show("a key that starts with a digit", {}, externals={**targets, "587722984435351614": target_orbit})
show("an external of the wrong shape for its kind", {}, externals={**targets, "target_orbit": target_orbit[:, :3]})
show("an external image that isn't the size of the image entries", {},
     externals={**targets, "target_image": target_image[:64, :64]})
show("a derived entry used before it's defined",
     {"derived": [{"name": "edges2", "method": "detect_edges", "inputs": ["drv.later"]}] + config["derived"]})
show("a derived entry of the wrong kind as weights",
     {"sim_metrics": [{**orbit_metric, "inputs": ["ext.target_orbit", "sim.search_orbit", "drv.target_edges"]}]})
show("a window that doesn't fit the target orbit",
     {"derived": [{**config["derived"][0], "params": {"k": 50.0}}] + config["derived"][1:]})
show("weights of the wrong length, known before the run",
     {"sim_metrics": [{**orbit_metric, "inputs": ["ext.target_orbit", "sim.search_orbit", "ext.short"]}]},
     externals={**targets, "short": np.ones(5)})
show("a metric with nothing that depends on the model",
     {"img_metrics": [{"method": "absolute_difference", "inputs": ["ext.target_image", "drv.target_edges"]}]})

A failure **during** the run stays where it happens. Below, each model gets a window made from its own
orbit. The second model is still approaching: its only pass is too long ago for the window to fit, so
its window can't be made and comes back as `None`. Its simulation and its metric are unaffected. (A
metric that used that window would fail for that model only, with a message naming `drv.model_win`.)

In [ ]:
approaching = params.copy()
approaching[3:6] *= -3
per_model = {
    "run": {"return": ["drv.model_win"]},
    "simulator": simulator,
    "derived": [{"name": "model_win", "method": "recent_approach_cosine_window", "inputs": ["sim.search_orbit"]}],
    "sim_metrics": [{"method": "closest_approach", "inputs": ["sim.search_orbit"], "select": "distance"}],
}
logging.disable(logging.CRITICAL)  # simr logs the failure with its traceback; quiet here
outcome = simr.run(np.stack([params, approaching]), per_model)
logging.disable(logging.NOTSET)
for m, label in enumerate(["the SDSS model", "still approaching"]):
    window_made = outcome.data["drv.model_win"][m, 0]
    print(f"{label}:")
    print("  simulation:", outcome.status["sim"][m, 0])
    print("  its window:", "made, " + str(window_made.shape) if window_made is not None else "None")
    print("  closest_approach:", outcome.status["sim_metrics"][m, 0, 0], "->", round(outcome.sim_metrics[m, 0, 0], 3))

## The composite: one number per model
A `composite` section turns the scalar metrics into one number per model: the weighted mean of its
metric cells over both levels, `Σ w·m / Σ w`. The weights have one value per cell of a model
(`sim_weights`: simulator × `sim_metrics` entry; `img_weights`: simulator × image × `img_metrics`
entry), needn't sum to 1, and a level without entries has no key. Below, the old code's **50/50 mix of
the two orbit errors**, leaving the image metrics out with zeros:

In [ ]:
orbit_mix = simr.run(models, {**config, "composite": {"sim_weights": [[0.5, 0.5]], "img_weights": [[[0, 0, 0]]]}},
                     externals=targets)
print("weights used:", orbit_mix.composite_weights)
print()
print(f"{'mass2 x':>8} {'orbit time':>11} {'orbit position':>15} {'composite':>10}")
for m, factor in enumerate(factors):
    time_error, position_error = orbit_mix.sim_metrics[m, 0]
    print(f"{factor:>8} {time_error:>11.4f} {position_error:>15.4f} {orbit_mix.composite[m]:>10.4f}")

The math is one function, `metrics.composite`, so other weights can be tried on returned arrays without
simulating again. Raw values differ in scale (orbit errors here are much larger than image
differences), so a mix is dominated by its largest terms unless the weights compensate:

In [ ]:
for label, weights in {
    "orbits only, 50/50": {"sim_weights": [[0.5, 0.5]], "img_weights": [[[0, 0, 0]]]},
    "everything, equal": {"sim_weights": [[1, 1]], "img_weights": [[[1, 1, 1]]]},
    "orbits + tidal-weighted image x 20": {"sim_weights": [[0.5, 0.5]], "img_weights": [[[0, 20, 0]]]},
}.items():
    values = metrics.composite(full_run.sim_metrics, full_run.img_metrics, **weights)
    print(f"{label:<36}", "  ".join(f"{value:7.4f}" for value in values))

- `"composite": {}` weighs every entry that gives one float equally, and descriptive entries (dicts) 0.
- A descriptive entry must have weight 0; a failed or skipped cell with a nonzero weight makes only
  that model's composite NaN (a failed scalar cell holds NaN).
- Without the section (or with `null`) there is no composite: `result.composite` is `None`.

## On record: saving, and running again
With an output, the run's record in the catalog holds the metric values and the composite;
`simr.load_metrics` reads them back. `run.save` also takes externals (`ext.<key>`) and derived entries
(`drv.<name>`): an external, or an entry made only from externals, is stored once for the output, named
by its hash; an entry made from a model's data goes in that model's folder.

In [ ]:
import tempfile
from pathlib import Path

output = Path(tempfile.mkdtemp()) / "runs"
on_record = {**config, "run": {"save": ["ext.target_orbit", "ext.target_image", "drv.target_win", "drv.tidal"]},
             "composite": {"sim_weights": [[0.5, 0.5]], "img_weights": [[[0, 0, 0]]]}}
saved_run = simr.run(models[:2], on_record, externals=targets, output=output)
for path in sorted(output.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(output).as_posix():<92} {path.stat().st_size:>8,} bytes")
print()
values = simr.load_metrics(output)
print("composite, returned:", saved_run.composite)
print("composite, loaded:  ", values["composite"])
print("a model's saved derived arrays:", list(simr.load_derived(output / "model_000000")))

`simr.load_run` rebuilds a recorded run's inputs (its models, full config and externals) under
`simr.run`'s own argument names, so the run can be repeated from the catalog alone. Here the externals
were given in memory, and come back from the copies `run.save` made. Without those copies, `load_run`
says which externals to give it.

In [ ]:
inputs = simr.load_run(output)
print({key: type(value).__name__ for key, value in inputs.items()})
print("externals:", {key: Path(value["path"]).parent.name + "/" + Path(value["path"]).name[:12] + "..."
                     for key, value in inputs["externals"].items()})
again = simr.run(**inputs, output=output)
print("same composite:", np.array_equal(again.composite, saved_run.composite))

## What's next
Still to come in MDRN-4: the first image metrics beyond `absolute_difference`, a run plan with a memory
cap, and a first orbit-only run from the command line. Reusing saved metric values, and a standard way
to scale raw metrics, wait until there are more metrics to need them.